In [0]:
%run ./env

In [0]:
%run ./python_libraries

In [0]:
%run ../delta_function

In [0]:
%run ./delta_schema

In [0]:
%run ./load_data

In [0]:
%run ./transform_data

In [0]:
%run ./dim_batch

In [0]:
%run ./dim_parameter

In [0]:
%run ./dim_report_parameter_mapping

In [0]:
%run ./fact_batch_measures

In [0]:
# Controle final : l'equipe energie peut modifier sa configuration sans prevenir.
# Le job a deja tout charge a ce stade ; on le fait echouer pour declencher la
# notification d'echec Databricks, sans bloquer le chargement des indicateurs.
# Seule exception voulue : un simple renommage de code (meme id) ne fait pas
# echouer le job, le rapport continue de fonctionner par l'id.
mapping_cible = spark.table(current_catalog + "." + current_schema + ".dim_report_parameter_mapping")

# Parametre du rapport supprime ou introuvable dans la source
anomalies_codes = (
    mapping_cible.alias("m")
    .join(
        parameters_variables.alias("pv"),
        F.col("m.parameter_id") == F.col("pv.id_parameter_variable"),
        "left"
    )
    .filter(
        F.col("pv.id_parameter_variable").isNull()
        | F.coalesce(F.col("pv.deleted"), F.lit(False))
    )
    .select(
        F.col("m.display_order"),
        F.col("m.column_label"),
        F.col("m.code").alias("code_mapping"),
        F.col("pv.code").alias("code_source"),
        F.col("pv.deleted").alias("supprime_source")
    )
)

# Parametre automatique dont le code actuel dans la source est absent
# d'energy_metadata : ses mesures ne sont plus chargees. On part du code de la
# source (par l'id) et non de celui du mapping, pour qu'un renommage coherent
# des deux cotes ne declenche rien.
anomalies_metadata = (
    mapping_cible.alias("m")
    .join(
        parameters_variables.alias("pv"),
        F.col("m.parameter_id") == F.col("pv.id_parameter_variable"),
        "inner"
    )
    .filter(
        ~F.col("m.code").isin(codes_manuels_attendus)
        & ~F.col("pv.code").isin(codes_manuels_attendus)
    )
    .join(
        energy_metadata.select(F.col("measure_name").alias("code_metadata")).distinct(),
        F.col("pv.code") == F.col("code_metadata"),
        "left_anti"
    )
    .select(
        F.col("m.display_order"),
        F.col("m.column_label"),
        F.col("m.code").alias("code_mapping"),
        F.col("pv.code").alias("code_source")
    )
)

nb_anomalies_codes = anomalies_codes.count()
nb_anomalies_metadata = anomalies_metadata.count()

if nb_anomalies_codes > 0:
    print(f"ATTENTION : {nb_anomalies_codes} indicateur(s) du rapport supprime(s) ou introuvable(s) dans la source :")
    anomalies_codes.orderBy("display_order").show(truncate=False)

if nb_anomalies_metadata > 0:
    print(f"ATTENTION : {nb_anomalies_metadata} indicateur(s) du rapport absents d'energy_metadata, leurs mesures ne sont plus chargees :")
    anomalies_metadata.orderBy("display_order").show(truncate=False)

# Photo de la configuration energie de la ligne cote source (energy_metadata de la
# ligne + parametres associes dans parameters_variables). Toute difference avec la
# photo du run precedent fait echouer le job : mesure ajoutee ou retiree,
# regle de calcul ou unite modifiee, bascule vers un autre id... La photo est mise
# a jour avant l'echec, chaque changement n'est donc signale qu'une fois.
# Les lignes sont reperees par l'id du parametre et le code n'est pas compare :
# un renommage fait des deux cotes (meme id) ne fait pas echouer le job.
# Les dates techniques sont exclues : energy_metadata est recreee chaque semaine.
ligne_controlee = "rouen1"
table_photo_config = current_catalog + "." + current_schema + ".controle_config_source"

energy_metadata_ligne = energy_metadata.filter(F.col("prd_line") == ligne_controlee)
parametres_actifs = parameters_variables.filter(~F.coalesce(F.col("deleted"), F.lit(False)))
colonnes_metadata = [c for c in energy_metadata.columns if c not in ("prd_line", "measure_name", "created_at", "modified_at")]
colonnes_parametre = ["unit", "phase", "activity", "task", "sequence", "category", "manual_entry", "deleted", "description"]

# Mesure sans parametre correspondant dans la source : reperee par son code
config_metadata = (
    energy_metadata_ligne.alias("em")
    .join(
        parametres_actifs.alias("pv"),
        F.col("em.measure_name") == F.col("pv.code"),
        "left"
    )
    .select(
        F.lit("energy_metadata").alias("objet"),
        F.coalesce(
            F.col("pv.id_parameter_variable").cast("string"),
            F.concat(F.lit("code:"), F.col("em.measure_name"))
        ).alias("cle"),
        F.to_json(F.struct(*[F.col("em." + c).alias(c) for c in colonnes_metadata])).alias("contenu")
    )
)

# Parametres suivis : ceux du rapport et ceux declares pour la ligne dans energy_metadata
ids_rapport = mapping_cible.select(F.col("parameter_id").cast("long").alias("id"))
ids_metadata = (
    parametres_actifs
    .join(
        energy_metadata_ligne.select(F.col("measure_name").alias("code")).distinct(),
        on="code",
        how="inner"
    )
    .select(F.col("id_parameter_variable").cast("long").alias("id"))
)
ids_rapport.createOrReplaceTempView("controle_ids_rapport")
ids_metadata.createOrReplaceTempView("controle_ids_metadata")
ids_suivis = spark.sql("SELECT id FROM controle_ids_rapport UNION SELECT id FROM controle_ids_metadata")

config_parametres = (
    parameters_variables
    .join(ids_suivis, F.col("id_parameter_variable").cast("long") == F.col("id"), "inner")
    .select(
        F.lit("parameters_variables").alias("objet"),
        F.col("id_parameter_variable").cast("string").alias("cle"),
        F.to_json(F.struct(*colonnes_parametre)).alias("contenu")
    )
)

# Union via SQL (unionByName appelle _jdf, non supporte sur cluster partage Unity Catalog)
config_metadata.createOrReplaceTempView("controle_config_metadata")
config_parametres.createOrReplaceTempView("controle_config_parametres")
config_actuelle = spark.sql("""
    SELECT objet, cle, contenu FROM controle_config_metadata
    UNION ALL
    SELECT objet, cle, contenu FROM controle_config_parametres
""")

nb_changements_config = 0
if spark.catalog.tableExists(table_photo_config):
    photo_precedente = spark.table(table_photo_config)
    lignes_avant = photo_precedente.join(config_actuelle, ["objet", "cle", "contenu"], "left_anti")
    lignes_maintenant = config_actuelle.join(photo_precedente, ["objet", "cle", "contenu"], "left_anti")
    nb_changements_config = lignes_avant.count() + lignes_maintenant.count()

    # Affichage avant l'ecrasement de la photo, qui est relue par ces DataFrames
    if nb_changements_config > 0:
        print(f"ATTENTION : la configuration energie de {ligne_controlee} a change cote source depuis le dernier run.")
        print("AVANT (lignes modifiees ou retirees) :")
        lignes_avant.orderBy("objet", "cle").show(truncate=False)
        print("MAINTENANT (lignes modifiees ou ajoutees) :")
        lignes_maintenant.orderBy("objet", "cle").show(truncate=False)
else:
    print(f"{table_photo_config} : premiere photo de la configuration, aucune comparaison ce run")

config_actuelle.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_photo_config)

if nb_anomalies_codes + nb_anomalies_metadata + nb_changements_config > 0:
    raise Exception(
        "Parametres energie modifies cote source : verifier la liste de "
        "dim_report_parameter_mapping (detail dans la sortie ci-dessus). "
        "Les donnees ont bien ete chargees."
    )